# Hunyuan3D 2.1 — Kaggle Weights Dataset Builder

This notebook downloads **only the model assets required by the Clash of Legends generation notebook** and arranges them in the exact directory layout that notebook detects.

**Output folder:** `/kaggle/working/hunyuan3d-weights/`

After a successful run, use Kaggle's **Save Version → Save & Run All** and create a Kaggle Dataset from the notebook output. Attach that Dataset to the generation notebook with **Add Input → Your Datasets**.

Do not upload these model weights to the game's Hugging Face output Dataset. This is a separate Kaggle input Dataset used only to cache model weights.

License: Tencent Hunyuan3D 2.1 Community License applies to the Tencent model weights. Also review the DINOv2, Real-ESRGAN and U²-Net licenses before redistribution. This notebook is for personal preparation; do not redistribute weights unless each upstream license permits it.

In [ ]:
import os, shutil, subprocess, sys
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout or 'No GPU listing (GPU is not required for downloading weights).')
FREE = shutil.disk_usage('/kaggle/working').free / 2**30
print(f'Free disk: {FREE:.2f} GiB')
if FREE < 19.0:
    raise RuntimeError(f'Only {FREE:.1f} GiB free. The required model files are about 17.8 GiB; allow additional room for metadata and temporary downloads. Free more disk before continuing.')
OUT = '/kaggle/working/hunyuan3d-weights'
os.makedirs(OUT, exist_ok=True)
print('Preflight passed. Output:', OUT)

## Download the exact weight set

The target layout is deliberately identical to the generation notebook's `WEIGHTS` directory. The downloads are performed one repository at a time to avoid concurrent writes into the same Hugging Face cache.

In [ ]:
%pip -q install -U 'huggingface_hub>=0.30.2' hf_transfer
import os, shutil, urllib.request, time
from huggingface_hub import snapshot_download
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

def download_hub(repo_id, patterns):
    print(f'Downloading {repo_id} ...', flush=True)
    snapshot_download(repo_id=repo_id, allow_patterns=patterns, local_dir=os.path.join(OUT, repo_id))
    shutil.rmtree(os.path.join(OUT, repo_id, '.cache'), ignore_errors=True)
    print(f'Completed {repo_id}', flush=True)

def download_url(url, relpath):
    dest = os.path.join(OUT, relpath)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.isfile(dest) and os.path.getsize(dest) > 1024:
        print('Already present:', relpath); return
    part = dest + '.part'
    print('Downloading', relpath, flush=True)
    urllib.request.urlretrieve(url, part)
    if os.path.getsize(part) < 1024:
        raise RuntimeError(f'Download looks incomplete: {relpath}')
    os.replace(part, dest)

download_hub('tencent/Hunyuan3D-2.1', ['hunyuan3d-dit-v2-1/*', 'hunyuan3d-paintpbr-v2-1/*'])
download_hub('facebook/dinov2-giant', ['*.json', '*.safetensors', '*.txt'])
download_url('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth', 'RealESRGAN_x4plus.pth')
download_url('https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx', 'u2net/u2net.onnx')
print('Download stage finished.')

## Verify the complete directory structure

This check uses the exact five required entry points from the generation notebook. It also removes Hugging Face download-cache metadata so the resulting Dataset is smaller.

In [ ]:
import os, json
required = [
 'tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/model.fp16.ckpt',
 'tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1/unet/diffusion_pytorch_model.bin',
 'facebook/dinov2-giant/model.safetensors',
 'RealESRGAN_x4plus.pth',
 'u2net/u2net.onnx',
]
missing = [p for p in required if not os.path.isfile(os.path.join(OUT,p))]
if missing:
    raise RuntimeError('Missing required weight files:\n' + '\n'.join(missing))
for rel in required:
    p=os.path.join(OUT,rel); print(f'{os.path.getsize(p)/2**30:6.2f} GiB  {rel}')
total=sum(os.path.getsize(os.path.join(d,f)) for d,_,fs in os.walk(OUT) for f in fs)
print(f'\nVERIFIED: {total/2**30:.2f} GiB')
manifest={'layout_root':'', 'weights_root':'/kaggle/input/<your-weights-dataset>/hunyuan3d-weights', 'required_files':required, 'total_bytes':total}
with open(os.path.join(OUT,'WEIGHTS_MANIFEST.json'),'w') as f: json.dump(manifest,f,indent=2)
print('Output folder ready:', OUT)

## Publish as a Kaggle Dataset

1. Confirm the verification cell reports `VERIFIED` and all five files are present.
2. Choose **Save Version → Save & Run All**. Wait for the notebook version to finish successfully.
3. From the saved notebook output, create a new Kaggle Dataset named `hunyuan3d-2-1-weights` (or add the output folder to a Dataset with that name). Keep the folder `hunyuan3d-weights` at the Dataset root.
4. In the Clash of Legends generation notebook, choose **Add Input → Your Datasets** and attach that weights Dataset.
5. The generation notebook should then discover the nested `tencent/Hunyuan3D-2.1` folder and use the attached files rather than downloading the 17.8 GB again.

**Important:** this notebook does not create or publish a Kaggle Dataset automatically; Kaggle's Dataset creation/attachment is a UI action. It does not use `HF_TOKEN`, and it does not upload weights to Hugging Face.